# Ch 23 — 처음부터 vs 파인튜닝

from-scratch / continued pre-training / full SFT / LoRA 4가지 길의 결정 트리와 노트북에서 가능한 파인튜닝 크기 메모리 산수.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch23_from_scratch_vs_finetune.ipynb)

In [ ]:
# !pip install -q torch transformers peft datasets

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram:.1f} GB")
else:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f"RAM: {ram:.1f} GB")

## 2. 결정 트리

```
1. 도메인이 일반 (영어/한국어) 라 기성 모델로 충분?
   Yes → Ch 22 결정 트리로 → 끝 (LoRA 도 X)
   No  → 다음

2. 도메인 데이터 양?
   100B+ 토큰        → from-scratch (대형 GPU 클러스터 필요)
   1B~10B 토큰       → continued pre-training
   10K~1M 페어       → full SFT 또는 LoRA
   100~10K 페어      → LoRA / QLoRA

3. 노트북에서 끝낼 것?
   Yes → LoRA / QLoRA (Ch 24)
   No  → full SFT 가능 (단일 A100+)

4. 라이선스 분리 필요? (어댑터만 공유하고 싶은가)
   Yes → LoRA (어댑터만 별도 저장)
   No  → full SFT 도 OK
```

## 3. 노트북에서 가능한 파인튜닝 크기 (최소 예제)

Ch 11 의 메모리 식 14N 을 바탕으로 Full SFT / LoRA / QLoRA 메모리를 계산.

In [ ]:
# 파인튜닝 메모리 산수

def full_sft_memory_gb(params_b, batch=4, seq=1024):
    """Full SFT 메모리 추정 (GB). params_b: 파라미터 수 (단위: B)"""
    params = params_b * 1e9
    weights = params * 2 / 1e9       # bf16
    grads   = params * 2 / 1e9       # bf16
    adam    = params * 8 / 1e9       # fp32 x2
    act     = batch * seq * params_b * 0.001  # rough estimate
    return weights + grads + adam + act

def lora_memory_gb(params_b, r=16, batch=4, seq=1024):
    """LoRA 메모리 추정 (GB). base 동결, 어댑터만 학습."""
    params = params_b * 1e9
    base   = params * 2 / 1e9       # bf16 동결
    # LoRA params ≈ 1% of base
    lora_p = params * 0.01
    lora   = lora_p * (2 + 2 + 8) / 1e9  # params + grads + adam
    act    = batch * seq * params_b * 0.001
    return base + lora + act

def qlora_memory_gb(params_b, batch=4, seq=1024):
    """QLoRA 메모리 추정 (GB). base int4, LoRA bf16."""
    params = params_b * 1e9
    base   = params * 0.5 / 1e9    # int4 = 0.5 bytes
    lora_p = params * 0.01
    lora   = lora_p * (2 + 2 + 8) / 1e9
    act    = batch * seq * params_b * 0.001
    return base + lora + act

print("=== Full SFT ===")
for p in [0.5, 1.5, 3.0, 7.0]:
    print(f"  {p}B: {full_sft_memory_gb(p):.1f} GB")

print("\n=== LoRA (r=16) ===")
for p in [0.5, 1.5, 3.0, 7.0]:
    print(f"  {p}B: {lora_memory_gb(p):.1f} GB")

print("\n=== QLoRA (int4 base) ===")
for p in [7.0, 13.0, 70.0]:
    print(f"  {p}B: {qlora_memory_gb(p):.1f} GB")

if device == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n현재 VRAM: {vram:.1f} GB")
    print("LoRA 로 학습 가능한 최대 모델:")
    for p in [0.5, 1.5, 3.0, 7.0]:
        if lora_memory_gb(p) <= vram:
            print(f"  {p}B LoRA 가능")

## 4. 어느 길을 가나 — 본 책 캡스톤의 결정 (실전)

In [ ]:
# 캡스톤 결정 트리 실행

print("=== 캡스톤 (한국 동화 생성기) 결정 트리 ===")
print()
print("Q1. 일반 한국어 모델로 충분?")
print("  → No (동화 도메인 특화 필요)")
print()
print("Q2. 데이터 양?")
print("  → 5K~50K 합성 동화 페어 → LoRA / QLoRA")
print()
print("Q3. 노트북에서 끝낼 것?")
print("  → Yes → LoRA / QLoRA")
print()
print("Q4. 라이선스 분리?")
print("  → Yes (어댑터만 HF Hub 업로드)")
print()
print("결론: LoRA on Qwen 2.5-0.5B-Instruct")
print()

# 0.5B LoRA 메모리 확인
mem = lora_memory_gb(0.5)
print(f"Qwen 2.5-0.5B LoRA 예상 메모리: {mem:.1f} GB")
print("T4 (16 GB): 충분히 가능")

## 연습문제

1. 본인 도메인 데이터 양을 측정 (페어 또는 토큰 수). 4가지 길 중 어디?
2. 0.5B / 1.5B / 3B 의 LoRA 메모리를 본인 GPU 에 맞춰 산수.
3. SmolLM2-360M 에 100 페어 LoRA vs full SFT 비교 (가능하면). 결과 차이?
4. **(생각해볼 것)** 본 책 10M 모델을 from-scratch 한 경험이 1.5B LoRA 결정에 어떤 영향을 주는가? "처음부터 만든 사람" 이 가지는 직관은?

In [ ]:
# 연습 1: 본인 도메인 데이터 양 측정 + 결정 트리


In [ ]:
# 연습 2: 본인 GPU 에 맞춰 메모리 산수
# lora_memory_gb(0.5), lora_memory_gb(1.5), lora_memory_gb(3.0) 결과 확인
for p in [0.5, 1.5, 3.0]:
    print(f"{p}B LoRA: {lora_memory_gb(p):.1f} GB")


In [ ]:
# 연습 3: SmolLM2-360M LoRA vs full SFT 비교
